# **Semana 11: NoSQL - Redis y Neo4j (NB2 - Práctico/Ejercicios)**

## **Propósito de la Sesión**
Explorar dos tipos adicionales de bases de datos NoSQL: **Redis** (clave-valor en memoria) y **Neo4j** (grafos). Aprenderemos a configurar instancias gratuitas en la nube, conectarnos desde Python y realizar operaciones características de cada motor.

### **Objetivos de Aprendizaje**
Al finalizar este notebook, el estudiante será capaz de:
1. **Crear** una base de datos Redis gratuita en Redis Cloud.
2. **Conectar** desde Python a Redis y realizar operaciones de caché (set/get, expiración).
3. **Crear** un sandbox gratuito en Neo4j AuraDB.
4. **Conectar** desde Python a Neo4j usando `py2neo` y ejecutar consultas Cypher.
5. **Comparar** los casos de uso de cada tipo de base de datos NoSQL.

## **Configuración Inicial**

Instalaremos las librerías necesarias para trabajar con Redis y Neo4j.

In [ ]:
# Instalación de librerías
!pip install redis pandas matplotlib seaborn --quiet
!pip install py2neo --quiet

# Importaciones
import redis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json
from datetime import datetime
from py2neo import Graph, Node, Relationship
from getpass import getpass

# Configuración de visualización
sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)

print("Librerías instaladas e importadas correctamente.")

---
## **Parte 1: Redis Cloud - Base de datos clave-valor en memoria**

### **¿Qué es Redis?**

Redis es una base de datos en memoria, de tipo clave-valor, extremadamente rápida. Soporta estructuras de datos como strings, hashes, listas, sets y sorted sets.

### **Casos de uso:**
*   **Caché**: Almacenar resultados de consultas costosas.
*   **Sesiones de usuario**: Mantener estado en aplicaciones web.
*   **Rate limiting**: Controlar número de peticiones.
*   **Colas de tareas**: Con listas.
*   **Leaderboards**: Con sorted sets.
*   **IA**: Caché de predicciones, almacenamiento de embeddings.

### **1.1. Crear una base de datos Redis gratuita en Redis Cloud**

Sigue estos pasos para configurar tu Redis en la nube:

1. **Registro**:
   *   Ve a [https://redis.com/try-free/](https://redis.com/try-free/).
   *   Crea una cuenta gratuita (puedes usar Google o GitHub).

2. **Crear base de datos**:
   *   Una vez en el dashboard, haz clic en "Create Database".
   *   Selecciona "Free" (30 MB).
   *   Elige un nombre para tu base de datos (ej. "mi-redis-curso").
   *   Selecciona la región más cercana.
   *   Haz clic en "Create".

3. **Obtener credenciales**:
   *   Una vez creada, verás los detalles de conexión:
     *   **Public endpoint**: `redis-xxxxx.region.redns.redis-cloud.com:xxxxx`
     *   **Default user**: `default`
     *   **Password**: La contraseña que configuraste (o la generada automáticamente).

✅ ¡Ya tienes tu Redis en la nube!

### **1.2. Conectar a Redis desde Python**

In [ ]:
# Ingresar credenciales de Redis de forma segura
print("--- CONEXIÓN A REDIS CLOUD ---")
redis_host = input("Host (ej. redis-xxxxx.region.redns.redis-cloud.com): ")
redis_port = input("Puerto (ej. 12345): ")
redis_password = getpass("Contraseña: ")

try:
    # Conectar a Redis
    r = redis.Redis(
        host=redis_host,
        port=int(redis_port),
        password=redis_password,
        decode_responses=True  # Para obtener strings en lugar de bytes
    )
    
    # Verificar conexión
    r.ping()
    print("✅ Conexión exitosa a Redis Cloud!")
    
    # Información del servidor
    info = r.info()
    print(f"Redis versión: {info['redis_version']}")
    print(f"Memoria usada: {int(info['used_memory'])/1024/1024:.2f} MB")
    
except Exception as e:
    print(f"❌ Error de conexión: {e}")

### **1.3. Operaciones básicas con Redis**

In [ ]:
print("--- OPERACIONES BÁSICAS CON REDIS ---")

# 1. SET y GET (strings)
print("\n1. SET/GET:")
r.set("nombre", "Ana Pérez")
r.set("edad", 30)
print(f"  nombre: {r.get('nombre')}")
print(f"  edad: {r.get('edad')}")

# 2. Incrementar contadores
print("\n2. Contadores (INCR):")
r.set("contador", 0)
r.incr("contador")
r.incr("contador")
r.incrby("contador", 5)
print(f"  contador: {r.get('contador')}")

# 3. Trabajar con listas
print("\n3. Listas (LPUSH, LRANGE):")
r.lpush("tareas", "estudiar")
r.lpush("tareas", "programar")
r.rpush("tareas", "dormir")
tareas = r.lrange("tareas", 0, -1)
print(f"  tareas: {tareas}")

# 4. Hashes (diccionarios)
print("\n4. Hashes (HSET, HGETALL):")
r.hset("usuario:1001", mapping={
    "nombre": "Carlos López",
    "email": "carlos@mail.com",
    "ciudad": "Madrid"
})
usuario = r.hgetall("usuario:1001")
print(f"  usuario: {usuario}")

# 5. Sets
print("\n5. Sets (SADD, SMEMBERS):")
r.sadd("etiquetas:producto:1", "electronica", "gaming", "laptop")
r.sadd("etiquetas:producto:2", "electronica", "mouse", "inalambrico")
etiquetas = r.smembers("etiquetas:producto:1")
print(f"  etiquetas producto 1: {etiquetas}")

# 6. Sorted Sets (para rankings)
print("\n6. Sorted Sets (ZADD, ZREVRANGE):")
r.zadd("ranking", {"jugador1": 100, "jugador2": 85, "jugador3": 120})
ranking = r.zrevrange("ranking", 0, -1, withscores=True)
print(f"  ranking: {ranking}")

# 7. Expiración (TTL)
print("\n7. Expiración (EXPIRE, TTL):")
r.setex("clave_temporal", 10, "Este mensaje se autodestruirá")
print(f"  TTL antes: {r.ttl('clave_temporal')} segundos")
time.sleep(2)
print(f"  TTL después de 2s: {r.ttl('clave_temporal')} segundos")
print(f"  Valor: {r.get('clave_temporal')}")
time.sleep(9)
print(f"  Después de 11s (debería haber expirado): {r.get('clave_temporal')}")

### **1.4. Caso práctico: Sistema de caché para consultas simuladas**

Simularemos un escenario donde tenemos consultas costosas a una base de datos. Usaremos Redis como caché para acelerar las respuestas.

In [ ]:
print("--- SIMULACIÓN DE CACHÉ CON REDIS ---")

def consulta_costosa(parametro):
    """Simula una consulta que tarda 2 segundos"""
    time.sleep(2)
    return f"Resultado para {parametro}: {parametro * 10}"

def obtener_datos(parametro):
    """Obtiene datos usando caché Redis"""
    clave = f"consulta:{parametro}"
    
    # Intentar obtener del caché
    resultado_cache = r.get(clave)
    
    if resultado_cache:
        print(f"  ✅ CACHE HIT - {clave}")
        return resultado_cache
    else:
        print(f"  ❌ CACHE MISS - {clave} (ejecutando consulta costosa)")
        resultado = consulta_costosa(parametro)
        # Guardar en caché por 30 segundos
        r.setex(clave, 30, resultado)
        return resultado

# Probar el sistema de caché
print("\nPrimera llamada (debería ser miss):")
start = time.time()
print(f"  Resultado: {obtener_datos(5)}")
print(f"  Tiempo: {time.time()-start:.2f}s")

print("\nSegunda llamada (debería ser hit):")
start = time.time()
print(f"  Resultado: {obtener_datos(5)}")
print(f"  Tiempo: {time.time()-start:.2f}s")

print("\nLlamada a otro parámetro (miss):")
start = time.time()
print(f"  Resultado: {obtener_datos(8)}")
print(f"  Tiempo: {time.time()-start:.2f}s")

---
## **Parte 2: Neo4j Sandbox - Base de datos de grafos**

### **¿Qué es Neo4j?**

Neo4j es una base de datos de grafos que almacena datos en nodos y relaciones, permitiendo consultas complejas sobre conexiones.

### **Casos de uso:**
*   **Redes sociales**: Amistades, seguidores.
*   **Sistemas de recomendación**: "A los clientes que compraron X también les gustó Y".
*   **Detección de fraude**: Identificar redes de transacciones sospechosas.
*   **Gestión de identidades**: Relaciones entre personas, empresas, direcciones.
*   **IA**: Grafos de conocimiento, análisis de caminos.

### **2.1. Crear un sandbox gratuito en Neo4j AuraDB**

Sigue estos pasos:

1. **Registro**:
   *   Ve a [https://neo4j.com/cloud/aura/](https://neo4j.com/cloud/aura/).
   *   Haz clic en "Start Free".
   *   Regístrate con tu correo o cuenta de Google.

2. **Crear instancia**:
   *   Una vez en el dashboard, haz clic en "Create Database".
   *   Selecciona "Free" (AuraDB Free).
   *   Elige un nombre para tu base de datos (ej. "grafos-curso").
   *   Selecciona la región más cercana.
   *   Establece una contraseña (guárdala bien).
   *   Haz clic en "Create Database".

3. **Obtener credenciales**:
   *   Espera a que la instancia se cree (unos minutos).
   *   Obtendrás una URI similar a: `neo4j+s://xxxxx.databases.neo4j.io`
   *   Usuario: `neo4j`
   *   Contraseña: la que estableciste.

✅ ¡Ya tienes tu base de datos de grafos en la nube!

### **2.2. Conectar a Neo4j desde Python con py2neo**

In [ ]:
# Ingresar credenciales de Neo4j
print("--- CONEXIÓN A NEO4J AURADB ---")
neo4j_uri = input("URI (ej. neo4j+s://xxxxx.databases.neo4j.io): ")
neo4j_user = input("Usuario (normalmente 'neo4j'): ") or "neo4j"
neo4j_password = getpass("Contraseña: ")

try:
    # Conectar a Neo4j
    graph = Graph(neo4j_uri, auth=(neo4j_user, neo4j_password))
    
    # Verificar conexión
    result = graph.run("RETURN 'Conexión exitosa' AS mensaje").data()
    print(f"✅ {result[0]['mensaje']} a Neo4j!")
    
except Exception as e:
    print(f"❌ Error de conexión: {e}")
    print("Verifica la URI, usuario y contraseña.")

### **2.3. Crear nodos y relaciones**

Construiremos un pequeño grafo social de personas y sus relaciones de amistad.

In [ ]:
print("--- CREANDO GRAFO SOCIAL ---")

# Limpiar la base de datos (opcional, tener cuidado)
# graph.run("MATCH (n) DETACH DELETE n")

# Crear nodos de personas
personas = [
    {"nombre": "Ana", "edad": 30},
    {"nombre": "Carlos", "edad": 32},
    {"nombre": "María", "edad": 28},
    {"nombre": "Juan", "edad": 35},
    {"nombre": "Laura", "edad": 27}
]

for p in personas:
    graph.run(
        "CREATE (n:Persona {nombre: $nombre, edad: $edad})",
        parameters=p
    )
print(f"✅ Creados {len(personas)} nodos Persona")

# Crear relaciones de amistad
amistades = [
    ("Ana", "Carlos"),
    ("Ana", "María"),
    ("Carlos", "Juan"),
    ("María", "Laura"),
    ("Juan", "Laura"),
    ("Laura", "Ana")
]

for a, b in amistades:
    graph.run(
        """
        MATCH (a:Persona {nombre: $a})
        MATCH (b:Persona {nombre: $b})
        CREATE (a)-[:AMIGO_DE]->(b)
        """,
        parameters={"a": a, "b": b}
    )
print(f"✅ Creadas {len(amistades)} relaciones AMIGO_DE")

### **2.4. Consultas Cypher**

Realizaremos consultas para explorar el grafo.

In [ ]:
print("--- CONSULTAS CYPHER ---")

# 1. Ver todos los nodos
print("\n1. Todos los nodos:")
result = graph.run("MATCH (n) RETURN n.nombre AS nombre, n.edad AS edad").data()
for row in result:
    print(f"  {row['nombre']} ({row['edad']} años)")

# 2. Ver relaciones de Ana
print("\n2. Amigos de Ana:")
result = graph.run(
    """
    MATCH (a:Persona {nombre: 'Ana'})-[:AMIGO_DE]->(amigo)
    RETURN amigo.nombre AS amigo
    """
).data()
for row in result:
    print(f"  {row['amigo']}")

# 3. Camino más corto entre María y Carlos
print("\n3. Camino más corto entre María y Carlos:")
result = graph.run(
    """
    MATCH p = shortestPath(
        (a:Persona {nombre: 'María'})-[:AMIGO_DE*]-(b:Persona {nombre: 'Carlos'})
    )
    RETURN [n in nodes(p) | n.nombre] AS camino
    """
).data()
if result:
    print(f"  {' -> '.join(result[0]['camino'])}")

# 4. Personas con más amigos
print("\n4. Personas con más amigos:")
result = graph.run(
    """
    MATCH (p:Persona)-[:AMIGO_DE]->(amigo)
    RETURN p.nombre AS persona, COUNT(amigo) AS num_amigos
    ORDER BY num_amigos DESC
    """
).data()
for row in result:
    print(f"  {row['persona']}: {row['num_amigos']} amigos")

# 5. Recomendación simple: amigos de amigos que no son amigos directos
print("\n5. Recomendaciones para Ana (amigos de amigos):")
result = graph.run(
    """
    MATCH (a:Persona {nombre: 'Ana'})-[:AMIGO_DE]->(amigo)-[:AMIGO_DE]->(recomendado)
    WHERE NOT (a)-[:AMIGO_DE]->(recomendado) AND a <> recomendado
    RETURN DISTINCT recomendado.nombre AS recomendacion
    """
).data()
for row in result:
    print(f"  {row['recomendacion']}")

---
## **Ejercicios Prácticos para el Estudiante**

Ahora aplica lo aprendido con estos ejercicios.

### **Ejercicio 1: Redis - Contador de visitas**

Crea un sistema simple que simule el contador de visitas a una página web:
1.  Cada vez que se ejecute una celda, debe incrementar un contador en Redis.
2.  Mostrar el número total de visitas.
3.  Añadir expiración: si pasan más de 60 segundos sin visitas, el contador se reinicia.

Ejecuta la celda varias veces para probarlo.

In [ ]:
# --- INICIO DE TU CÓDIGO ---

clave_visitas = "contador:visitas"

# Incrementar contador y establecer expiración
visitas = r.incr(clave_visitas)
r.expire(clave_visitas, 60)  # Expira después de 60 segundos sin actividad

print(f"👁️ Visita #{visitas} registrada")
print(f"⏱️ TTL restante: {r.ttl(clave_visitas)} segundos")

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 2: Redis - Caché de predicciones simuladas**

Imagina que tienes un modelo de ML que tarda en predecir. Implementa un sistema de caché con Redis que:
1.  Simule una predicción que tarda 3 segundos (con `time.sleep(3)`).
2.  Antes de ejecutar la predicción, verifique si ya existe en caché.
3.  Si existe, devuelva el resultado cacheado.
4.  Si no, ejecute la predicción, guárdela en caché y devuélvala.

Usa como clave el texto de entrada (ej. "cliente_123") y como valor la predicción simulada (ej. f"Predicción para {input}: 0.95").

In [ ]:
# --- INICIO DE TU CÓDIGO ---

def prediccion_simulada(entrada):
    """Simula una predicción que tarda 3 segundos"""
    time.sleep(3)
    return f"Predicción para {entrada}: {np.random.random():.3f}"

def obtener_prediccion(entrada):
    clave = f"prediccion:{entrada}"
    
    # Verificar caché
    resultado = r.get(clave)
    
    if resultado:
        print(f"✅ CACHE HIT para '{entrada}'")
        return resultado
    else:
        print(f"❌ CACHE MISS para '{entrada}' (calculando...)")
        resultado = prediccion_simulada(entrada)
        r.setex(clave, 30, resultado)  # Caché por 30 segundos
        return resultado

# Probar
print(obtener_prediccion("cliente_123"))
print(obtener_prediccion("cliente_123"))  # Debería ser instantáneo
print(obtener_prediccion("cliente_456"))

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 3: Neo4j - Sistema de recomendación simple**

Partiendo del grafo creado, añade:
1.  Nuevos nodos: "Producto" con nombre y precio.
2.  Relaciones: "COMPRÓ" entre personas y productos.
3.  Luego, escribe una consulta Cypher que recomiende productos a una persona basándose en lo que compraron sus amigos ("a los amigos de Ana les gusta...").

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# 1. Crear productos
productos = [
    {"nombre": "Laptop", "precio": 1200},
    {"nombre": "Mouse", "precio": 25},
    {"nombre": "Teclado", "precio": 80},
    {"nombre": "Monitor", "precio": 350},
    {"nombre": "Auriculares", "precio": 60}
]

for p in productos:
    graph.run(
        "CREATE (n:Producto {nombre: $nombre, precio: $precio})",
        parameters=p
    )
print("✅ Productos creados")

# 2. Crear relaciones de compra
compras = [
    ("Ana", "Laptop"),
    ("Ana", "Mouse"),
    ("Carlos", "Monitor"),
    ("Carlos", "Teclado"),
    ("María", "Auriculares"),
    ("Juan", "Monitor"),
    ("Laura", "Mouse"),
    ("Laura", "Auriculares")
]

for persona, producto in compras:
    graph.run(
        """
        MATCH (p:Persona {nombre: $persona})
        MATCH (prod:Producto {nombre: $producto})
        CREATE (p)-[:COMPRO]->(prod)
        """,
        parameters={"persona": persona, "producto": producto}
    )
print("✅ Relaciones de compra creadas")

# 3. Recomendación para Ana (productos que compraron sus amigos)
print("\n📌 Recomendaciones para Ana:")
result = graph.run(
    """
    MATCH (a:Persona {nombre: 'Ana'})-[:AMIGO_DE]->(amigo)-[:COMPRO]->(producto)
    WHERE NOT (a)-[:COMPRO]->(producto)
    RETURN DISTINCT producto.nombre AS producto, producto.precio AS precio
    ORDER BY producto.precio DESC
    """
).data()

for row in result:
    print(f"  {row['producto']} - ${row['precio']}")

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 4: Reflexión - Comparación de motores NoSQL**

Responde en una celda Markdown:

*   ¿En qué casos usarías Redis en lugar de Neo4j? ¿Y viceversa?
*   ¿Qué tipo de problemas se resuelven mejor con un grafo que con una base de datos clave-valor?
*   ¿Cómo se complementan Redis y Neo4j en una arquitectura de datos moderna?

**Tus respuestas:**

*   **Usaría Redis cuando...** 
    - Necesito alta velocidad y baja latencia (caché, sesiones, contadores).
    - Los datos tienen estructura simple (clave-valor, listas, sets).
    - Requiero expiración automática de datos.
    - Ejemplos: caché de consultas, rate limiting, leaderboards en tiempo real.

*   **Usaría Neo4j cuando...**
    - Los datos están altamente interconectados y las consultas dependen de las relaciones.
    - Necesito recorrer grafos (amistades, caminos, recomendaciones).
    - Las relaciones son tan importantes como los datos mismos.
    - Ejemplos: redes sociales, detección de fraude, motores de recomendación.

*   **Complementariedad:**
    - Redis puede servir como capa de caché para Neo4j, almacenando resultados de consultas frecuentes.
    - Neo4j maneja la complejidad relacional, mientras Redis proporciona acceso ultrarrápido a datos simples.
    - En una arquitectura moderna, pueden coexistir: Neo4j para el modelo de dominio relacional y Redis para mejorar el rendimiento.

---
## **Conclusiones de la Semana 11**

En este notebook hemos:

✅ **Configurado instancias gratuitas** de Redis y Neo4j en la nube.

✅ **Conectado desde Python** a ambos motores y realizado operaciones básicas.

✅ **Implementado un sistema de caché** con Redis, demostrando su utilidad para mejorar el rendimiento.

✅ **Creado un grafo social** en Neo4j y ejecutado consultas Cypher complejas (caminos, recomendaciones).

✅ **Comparado los casos de uso** de bases de datos clave-valor vs. grafos.

Estos conocimientos te permiten elegir la herramienta adecuada según el problema, e incluso combinarlas en arquitecturas multi-motor.

---
**Fin del Notebook - Semana 11**